# NBA DRAFT MODEL

In [1]:
# imports
import pandas as pd
import numpy as np
from scipy.stats import norm
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings("ignore")

In [2]:
# load df
df = pd.read_csv("nba_draft_model_data.csv")

In [3]:
df.columns

Index(['athlete_id', 'name_slug', 'season', 'season_label', 'team_id', 'team',
       'conference', 'name', 'position', 'games', 'starts', 'minutes',
       'points', 'turnovers', 'fouls', 'assists', 'steals', 'blocks', 'usage',
       'offensive_rating', 'defensive_rating', 'net_rating', 'porpag',
       'effective_field_goal_pct', 'true_shooting_pct',
       'assists_turnover_ratio', 'free_throw_rate', 'offensive_rebound_pct',
       'fg_pct', 'fg_attempted', 'fg_made', 'two_pt_fg_pct',
       'two_pt_fg_attempted', 'two_pt_fg_made', 'three_pt_fg_pct',
       'three_pt_fg_attempted', 'three_pt_fg_made', 'ft_pct', 'ft_attempted',
       'ft_made', 'rebounds_total', 'rebounds_defensive', 'rebounds_offensive',
       'win_shares_total_per40', 'win_shares_total', 'win_shares_defensive',
       'win_shares_offensive', 'hometown_state', 'hometown_city',
       'start_season', 'end_season', 'age', 'height_inches', 'weight_lbs',
       'draft_year', 'draft_round', 'overall_pick', 'total_game

In [4]:
# seperate data into two datasets

# continuous features to be z-scored (USING CAREER BPM NOT WS)
continuous_features = [
    'usage', 'offensive_rating', 'defensive_rating', 'porpag',
       'assists_turnover_ratio', 'free_throw_rate', 'offensive_rebound_pct',
       'fg_pct', 'two_pt_fg_pct', 'three_pt_fg_pct','ft_pct', 'rebounds_total',
       'win_shares_total_per40', 'age', 'height_inches', 'weight_lbs', 'overall_pick'
]

# create full feature list (can add in dummy variables in the future)
ALL_FEATURES = continuous_features

In [5]:
# create career bpm thresholds
THRESHOLDS = {
    "p_below_0":      (0.0,  "below"),   # bench player / out of the league
    "p_above_0":   (0.0,  "above"),   # decent starter or solid 6th man
    "p_above_2":  (2.0, "above"),   # good starter
    "p_above_4":  (4.0, "above"),   # consistent all-star consideration
    "p_above_6":  (6.0, "above"),   # an all-NBA career and more
}

# compute the probabilities using the thresholds
for col_name, (threshold, direction) in THRESHOLDS.items():
    if direction == "above":
        df[col_name] = 1 - norm.cdf(threshold, loc=df["career_bpm"], scale=df["standard_error"])
    else:
        df[col_name] = norm.cdf(threshold, loc=df["career_bpm"], scale=df["standard_error"])

In [6]:
# train/test split
train_df = df[df["career_bpm"].notna()].copy()
pred_df  = df[df["career_bpm"].isna()].copy()

print(f"Training players : {len(train_df)}")
print(f"2026 prospects   : {len(pred_df)}")

Training players : 488
2026 prospects   : 56


In [7]:
# scale features to account for size
scaler = StandardScaler()

X_train_cont = scaler.fit_transform(train_df[continuous_features].fillna(train_df[continuous_features].median()))
X_pred_cont  = scaler.transform(pred_df[continuous_features].fillna(train_df[continuous_features].median()))

X_train = np.hstack([X_train_cont])
X_pred  = np.hstack([X_pred_cont])

In [8]:
# HYPERPARAMETER TUNING
tscv = TimeSeriesSplit(n_splits=5)

# parameter grid to search
param_grid = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

best_params_per_target = {}

for target_col in THRESHOLDS.keys():
    y = train_df[target_col].values

    search = GridSearchCV(
        XGBRegressor(random_state=42),
        param_grid,
        scoring=mae_scorer,
        cv=tscv,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y)

    best_params_per_target[target_col] = search.best_params_
    print(f"\n── {target_col} ──────────────────────")
    print(f"  Best MAE : {-search.best_score_:.4f}")
    print(f"  Params   : {search.best_params_}")

Fitting 5 folds for each of 432 candidates, totalling 2160 fits

── p_below_0 ──────────────────────
  Best MAE : 0.3341
  Params   : {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'subsample': 1.0}
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

── p_above_0 ──────────────────────
  Best MAE : 0.3340
  Params   : {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'subsample': 1.0}
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

── p_above_2 ──────────────────────
  Best MAE : 0.1008
  Params   : {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 300, 'subsample': 1.0}
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

── p_above_4 ──────────────────────
  Best MAE : 0.0421
  Params   : {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0}
Fitting 5 folds for each of 432 candidates, tota

In [9]:
# train the 5 xgboost models using what we learned above
xgb_params = dict(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

models = {}
predictions = {}

for target_col in THRESHOLDS.keys():
    y = train_df[target_col].values
    model = XGBRegressor(**best_params_per_target[target_col], random_state=42)
    model.fit(X_train, y)
    models[target_col] = model
    predictions[target_col] = model.predict(X_pred)
    print(f"Trained model for {target_col}")

Trained model for p_below_0
Trained model for p_above_0
Trained model for p_above_2
Trained model for p_above_4
Trained model for p_above_6


In [10]:
# build results table
results = pred_df[["name", "overall_pick", "height_inches"]].copy().reset_index(drop=True)

# gather the raw predictions
p_below_0 = np.clip(predictions["p_below_0"], 0, 1)
p_above_0 = np.clip(predictions["p_above_0"], 0, 1)
p_above_2 = np.clip(predictions["p_above_2"], 0, 1)
p_above_4 = np.clip(predictions["p_above_4"], 0, 1)
p_above_6 = np.clip(predictions["p_above_6"], 0, 1)

total = p_below_0 + p_above_0 + p_above_2 + p_above_4 + p_above_6

results["tier_bench"]    = p_below_0 / total
results["tier_rotation"] = p_above_0 / total
results["tier_starter"]  = p_above_2 / total
results["tier_allstar"]  = p_above_4 / total
results["tier_alltime"]  = p_above_6 / total

# composite score calculation adjusted for the 0, 2, 4, 6 thresholds
results["composite_score"] = (
    0.20 * p_above_0 +
    0.20 * p_above_2 +
    0.20 * p_above_4 +
    0.20 * p_above_6 -
    0.20 * p_below_0
)

# sort and reset 1-based indexing for the draft board
results = results.sort_values("composite_score", ascending=False).reset_index(drop=True)
results.index += 1

In [11]:
#feature importance
print("\n Feature Importances for Bench Players (P < 0 model)")
feat_names = continuous_features
imp = pd.Series(models["p_below_0"].feature_importances_, index=feat_names)
print(imp.sort_values(ascending=False).head(10).to_string())

print("\n Feature Importances for Rotational Players(P > 0 model)")
feat_names = continuous_features
imp = pd.Series(models["p_above_0"].feature_importances_, index=feat_names)
print(imp.sort_values(ascending=False).head(10).to_string())

print("\n Feature Importances for Starters(P > 2 model)")
feat_names = continuous_features
imp = pd.Series(models["p_above_2"].feature_importances_, index=feat_names)
print(imp.sort_values(ascending=False).head(10).to_string())

print("\n Feature Importances for All-Stars (P > 4 model)")
feat_names = continuous_features
imp = pd.Series(models["p_above_4"].feature_importances_, index=feat_names)
print(imp.sort_values(ascending=False).head(10).to_string())

print("\n Feature Importances for All-Time Greats (P > 6 model)")
feat_names = continuous_features
imp = pd.Series(models["p_above_6"].feature_importances_, index=feat_names)
print(imp.sort_values(ascending=False).head(10).to_string())


 Feature Importances for Bench Players (P < 0 model)
overall_pick              0.128014
three_pt_fg_pct           0.078533
age                       0.073947
ft_pct                    0.067849
two_pt_fg_pct             0.065291
rebounds_total            0.064729
assists_turnover_ratio    0.063119
height_inches             0.060736
win_shares_total_per40    0.059646
fg_pct                    0.051976

 Feature Importances for Rotational Players(P > 0 model)
overall_pick              0.128014
three_pt_fg_pct           0.078533
age                       0.073947
ft_pct                    0.067849
two_pt_fg_pct             0.065291
rebounds_total            0.064729
assists_turnover_ratio    0.063119
height_inches             0.060736
win_shares_total_per40    0.059646
fg_pct                    0.051976

 Feature Importances for Starters(P > 2 model)
overall_pick              0.143192
porpag                    0.099715
fg_pct                    0.099526
win_shares_total_per40    0.090202


In [12]:
# vif
vif_df = train_df[continuous_features].dropna()
X_vif = vif_df.copy()
X_vif['intercept'] = 1

# calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["feature"] = continuous_features
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(continuous_features))]

# display sorted by highest VIF
print("\n" + "═" * 50)
print(f"  {'VARIANCE INFLATION FACTOR (VIF) SCORES':^44}")
print("═" * 50)
print(vif_data.sort_values(ascending=False, by="VIF").to_string(index=False, formatters={"VIF": "{:,.2f}".format}))
print("═" * 50)


══════════════════════════════════════════════════
     VARIANCE INFLATION FACTOR (VIF) SCORES   
══════════════════════════════════════════════════
               feature   VIF
                porpag 12.07
      offensive_rating 11.76
                fg_pct  8.29
         two_pt_fg_pct  5.20
                 usage  4.31
win_shares_total_per40  3.32
         height_inches  3.16
      defensive_rating  2.79
                ft_pct  2.71
assists_turnover_ratio  2.68
            weight_lbs  2.60
        rebounds_total  2.25
 offensive_rebound_pct  2.01
       three_pt_fg_pct  1.45
       free_throw_rate  1.45
                   age  1.44
          overall_pick  1.38
══════════════════════════════════════════════════


In [13]:
# text draft board
pd.set_option("display.float_format", "{:.1%}".format)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)

print("\n" + "═" * 100)
print(f"  {'2026 NBA DRAFT MODEL PROJECTED BOARD':^92}")
print("═" * 100)
print(f"  {'#':<4} {'Name':<22} {'Mock':<6} "
      f"{'Bench':<9} {'Rotation':<11} {'Starter':<10} {'AllStar':<9} {'AllTime':<9}  {'Score'}")
print("─" * 100)

for rank, row in results.iterrows():
    print(f"  {rank:<4} {row['name']:<22} {int(row['overall_pick']):<6}"
          f"{row['tier_bench']:<9.1%} {row['tier_rotation']:<11.1%} "
          f"{row['tier_starter']:<10.1%} {row['tier_allstar']:<11.1%} {row['tier_alltime']:<9.1%} "
          f"{row['composite_score']:.1%}")

print("═" * 100)
print("\nKey: Bench Player = P(<0)   Rotation = P(>0)   Starter = P(>2)   AllStar = P(>4)  AllTime = P(>6) ")


════════════════════════════════════════════════════════════════════════════════════════════════════
                              2026 NBA DRAFT MODEL PROJECTED BOARD                            
════════════════════════════════════════════════════════════════════════════════════════════════════
  #    Name                   Mock   Bench     Rotation    Starter    AllStar   AllTime    Score
────────────────────────────────────────────────────────────────────────────────────────────────────
  1    Caleb Wilson           4     10.1%     52.2%       35.6%      2.0%        0.2%      25.6%
  2    AJ Dybantsa            1     21.4%     41.0%       22.6%      13.2%       1.8%      18.3%
  3    Darryn Peterson        2     18.3%     52.9%       12.0%      15.3%       1.6%      17.9%
  4    Kingston Flemings      7     11.8%     73.9%       13.4%      0.7%        0.1%      17.8%
  5    Cameron Boozer         3     19.8%     50.9%       29.2%      0.2%        0.0%      17.1%
  6    Darius Acuff

# TESTING ON OLDER CLASSES

In [14]:
# isolate the desired class
backtest_raw = df[df['draft_year'] == 2016].copy()

# train ONLY on other historical draft classes
train_df = df[(df["career_bpm"].notna()) & (df['draft_year'] != 2016)].copy()

print(f"Training players (Excluding Desired Year): {len(train_df)}")
print(f"Backtest prospects: {len(backtest_raw)}")

# fit the scaler on the clean training data
scaler = StandardScaler()
X_train_clean = scaler.fit_transform(train_df[continuous_features].fillna(train_df[continuous_features].median()))
X_class_clean  = scaler.transform(backtest_raw[continuous_features].fillna(train_df[continuous_features].median()))

# train the 5 models on data
models = {}
predictions_class = {}

for target_col in THRESHOLDS.keys():
    y_train = train_df[target_col].values

    model = XGBRegressor(**best_params_per_target[target_col])
    model.fit(X_train_clean, y_train)
    models[target_col] = model

    # predict for the unseen class
    predictions_class[target_col] = np.clip(model.predict(X_class_clean), 0, 1)

# rebuild the evaluation Table
results_class = backtest_raw[["name", "overall_pick", "career_bpm"]].copy()

p_below_0_class = predictions_class["p_below_0"]
p_above_0_class = predictions_class["p_above_0"]
p_above_2_class = predictions_class["p_above_2"]
p_above_4_class = predictions_class["p_above_4"]
p_above_6_class = predictions_class["p_above_6"]

total_class = p_below_0_class + p_above_0_class + p_above_2_class + p_above_4_class + p_above_6_class

results_class_table = pd.DataFrame({
    "Name": results_class["name"],
    "Pick": results_class["overall_pick"].astype(int),
    "Bench": p_below_0_class / total_class,
    "Rotation": p_above_0_class / total_class,
    "Starter": p_above_2_class / total_class,
    "AllStar": p_above_4_class / total_class,
    "AllTime": p_above_6_class / total_class,
    "Model_Score": (0.20 * p_above_0_class + 0.20 * p_above_2_class + 0.20 * p_above_4_class + 0.20 * p_above_6_class - 0.20 * p_below_0_class),
    "Actual_BPM": results_class["career_bpm"]
})

results_class_table = results_class_table.sort_values("Model_Score", ascending=False).reset_index(drop=True)
results_class_table.index += 1

# print the board
print("\n" + "═" * 110)
print(f"  {'DRAFT BOARD':^100}")
print("═" * 110)
print(f"  {'#':<4} {'Name':<22} {'Pick':<6} {'Bench':<8} {'Rotation':<10} {'Starter':<9} {'AllStar':<9} {'AllTime':<9} {'Score':<8} {'Actual BPM'}")
print("─" * 110)

for rank, row in results_class_table.iterrows():
    print(f"  {rank:<4} {row['Name']:<22} {row['Pick']:<6} "
          f"{row['Bench']:<8.1%} {row['Rotation']:<10.1%} {row['Starter']:<9.1%} "
          f"{row['AllStar']:<9.1%} {row['AllTime']:<9.1%} {row['Model_Score']:<8.1%} {row['Actual_BPM']:.2f}")
print("═" * 110)

Training players (Excluding Desired Year): 463
Backtest prospects: 25

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                              DRAFT BOARD                                             
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  #    Name                   Pick   Bench    Rotation   Starter   AllStar   AllTime   Score    Actual BPM
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    Pascal Siakam          27     23.0%    17.9%      1.5%      37.5%     20.1%     26.4%    1.48
  2    Ben Simmons            1      19.2%    36.6%      43.9%     0.0%      0.3%      22.1%    2.41
  3    Jaylen Brown           3      30.9%    31.9%      29.5%     4.1%      3.6%      12.1%    0.44
  4    Jakob Poeltl           9      25.3%    70.8%      2.7%      1.0%      0.2%  

# ALL TIME PROSPECTS

In [15]:
# get the highest ranked prospects of all time
# isolate historical records with valid career data
all_historical = df[df["career_bpm"].notna()].copy()

# setup arrays to hold Out-Of-Fold (OOF) predictions
oof_predictions = {target: np.zeros(len(all_historical)) for target in THRESHOLDS.keys()}

# stratify or split by a standard KFold to get un-biased predictions across history
kf = KFold(n_splits=5, shuffle=True, random_state=42)
historical_features = all_historical[continuous_features].values

for train_idx, val_idx in kf.split(historical_features):
    X_tr, X_val = historical_features[train_idx], historical_features[val_idx]

    # scale within fold to avoid data leakage
    fold_scaler = StandardScaler()
    X_tr_scaled = fold_scaler.fit_transform(pd.DataFrame(X_tr).fillna(all_historical[continuous_features].median()))
    X_val_scaled = fold_scaler.transform(pd.DataFrame(X_val).fillna(all_historical[continuous_features].median()))

    # fit models and predict for this fold's validation set
    for target_col in THRESHOLDS.keys():
        y_tr = all_historical[target_col].iloc[train_idx].values

        # use grid-searched optimal parameters
        model = XGBRegressor(**best_params_per_target[target_col])
        model.fit(X_tr_scaled, y_tr)

        # clip to ensure valid probabilities
        pred_val = np.clip(model.predict(X_val_scaled), 0, 1)
        oof_predictions[target_col][val_idx] = pred_val

# build all-time leaderboard table
all_time_leaderboard = all_historical[["name", "draft_year", "overall_pick", "career_bpm"]].copy().reset_index(drop=True)

p_below_0_at = oof_predictions["p_below_0"]
p_above_0_at = oof_predictions["p_above_0"]
p_above_2_at = oof_predictions["p_above_2"]
p_above_4_at = oof_predictions["p_above_4"]
p_above_6_at = oof_predictions["p_above_6"]

total_at = p_below_0_at + p_above_0_at + p_above_2_at + p_above_4_at + p_above_6_at

all_time_leaderboard["Bench"] = p_below_0_at / total_at
all_time_leaderboard["Rotation"] = p_above_0_at / total_at
all_time_leaderboard["Starter"] = p_above_2_at / total_at
all_time_leaderboard["AllStar"] = p_above_4_at / total_at
all_time_leaderboard["AllTime"] = p_above_6_at / total_at

# apply composite weight logic
all_time_leaderboard["Model_Score"] = (
    0.20 * p_above_0_at +
    0.20 * p_above_2_at +
    0.20 * p_above_4_at +
    0.20 * p_above_6_at -
    0.20 * p_below_0_at
)

# format results
all_time_leaderboard = all_time_leaderboard.sort_values("Model_Score", ascending=False).reset_index(drop=True)
all_time_leaderboard.index += 1

# display Top 25 Prospects of All Time
pd.set_option("display.float_format", "{:.1%}".format)
print("\n" + "═" * 115)
print(f"  {'NBA DRAFT MODEL HIGHEST RATED PROSPECTS OF ALL TIME':^105}")
print("═" * 115)
print(f"  {'#':<4} {'Name':<22} {'Year':<6} {'Pick':<6} "
      f"{'Bench':<9} {'Rotation':<11} {'Starter':<10} {'AllStar':<9} {'AllTime':<9} | {'Score':<8} {'Actual BPM'}")
print("─" * 115)

for rank, row in all_time_leaderboard.head(25).iterrows():
    print(f"  {rank:<4} {row['name']:<22} {int(row['draft_year']):<6} {int(row['overall_pick']):<6}"
          f"{row['Bench']:<9.1%} {row['Rotation']:<11.1%} "
          f"{row['Starter']:<10.1%} {row['AllStar']:<11.1%} {row['AllTime']:<9.1%} | "
          f"{row['Model_Score']:<8.1%} {row['career_bpm']:.2f}")

print("═" * 115)


═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
                             NBA DRAFT MODEL HIGHEST RATED PROSPECTS OF ALL TIME                           
═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  #    Name                   Year   Pick   Bench     Rotation    Starter    AllStar   AllTime   | Score    Actual BPM
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1    Blake Griffin          2009   1     2.6%      33.5%       29.9%      30.0%       4.0%      | 52.5%    2.39
  2    Ja Morant              2019   2     0.0%      39.9%       20.3%      35.2%       4.6%      | 50.1%    2.24
  3    Kevin Love             2008   5     7.4%      27.5%       30.8%      32.9%       1.4%      | 48.8%    2.42
  4    Zach Edey              2024   9     10.4%     25.8%       24.2%      34.4% 